# Offline AI: From Private Chat to Coding and Agents

A practical, **offline-first** introduction to running AI models locally, asking questions about local documents, using AI in VS Code, and (optionally) giving an agent controlled access to a working folder.

**Target environment:** Linux. The original author tested parts of this guide on a machine with 16 GB RAM and a 6 GB NVIDIA GPU. Other hardware and software versions may behave differently; this notebook is not a guarantee of performance on every laptop.

## Learning outcomes
By the end, you should be able to:
1. Run a local language model with Ollama.
2. use Open WebUI for local chat and, after configuring local document processing, test questions about a document;
3. use Python for repeatable business calculations and ask an AI to explain the results;
4. optionally connect a local model to VS Code and test a restricted agent workflow.

### How to use this notebook
- Commands in fenced `bash` blocks are for a **terminal**, not a Python notebook cell.
- Python cells run in the notebook kernel. Run the dependency-install cell before cells that import `pandas`.
- Run cells in order. Do not paste secrets, passwords, customer data, or patient data into demos.
- **Prepare before going offline:** install software, download model weights, and install Python packages while internet access is available. Then disconnect and test the exact workflow you plan to demonstrate.

**Structure:** Part A is the audience guide. Part B contains presenter notes and should be removed or shared separately from the beginner version.


# PART A: Guide for the audience

## 1. The architecture

| Layer | Tool | Job |
|---|---|---|
| **Model runtime** | Ollama | Downloads and runs language models locally |
| **Chat/document interface** | Open WebUI | Browser-based chat; document retrieval requires its own local configuration |
| **Coding interface** | VS Code + Continue | Helps explain, draft, and edit code |
| **Agent layer** | OpenClaw | Connects a model to tools that can take actions, subject to its configuration and permissions |
| **Reliable calculations** | Python + pandas | Computes figures reproducibly instead of relying on an LLM's arithmetic |

A chat model mainly responds to prompts. An agent can call tools, inspect results, and continue through multiple steps. That extra capability introduces latency and risk, so start with chat and add agent access only when needed.

> **The golden rule: Python calculates. AI explains. You verify.**

### What “offline” means here
A local model can generate responses without sending the prompt to a cloud model. That does **not automatically prove the whole application is offline**: document parsing, embeddings, web search, extensions, updates, or other integrations may use additional services. Download all required components first, disable cloud features for the test, disconnect the network, and verify each part separately.


## 2. Choose your level

| Level | Outcome | Approx. time once prepared | Practical guidance |
|---|---|---:|---|
| **1 — Local chat** | Chat with a downloaded model; optionally test local document retrieval | 15 min | 8 GB RAM may run a small quantized model, but performance and document workflows vary. 16 GB is a better workshop target. |
| **2 — Coding help** | Use the local model in VS Code | +5–10 min | Test the extension and its configuration before the event. |
| **3 — Agent** | Let an agent perform a limited task in a dedicated folder | +20 min or more | Advanced/optional. Use a disposable workspace and verify every action. |

**Start at Level 1.** Do not make Level 3 a prerequisite for learning local AI. Actual model speed and memory use depend on model size, quantization, context length, runtime, and hardware.


## 3. Level 1: Ollama + Open WebUI

For the full Linux installation instructions, follow [Install Ollama](docs/setup/OLLAMA.md), [Install Open WebUI](docs/setup/OPEN_WEBUI.md), and [Connect Ollama to Open WebUI](docs/connect/OLLAMA_OPEN_WEBUI.md).

Quick start after completing those setup steps:

```bash
ollama pull llama3.2:3b
ollama run llama3.2:3b
```

Open WebUI at <http://127.0.0.1:3000>, select the local model, and try a short question. For private-document questions, use [Demo 02](docs/demos/02-document-questions.md). Local chat does not by itself guarantee that document extraction, embeddings, updates, or other integrations run locally; prepare and test those parts separately before demonstrating offline use.

In [ ]:
# Check that this notebook kernel has the project's Python dependency.
try:
    import pandas as notebook_pandas
    print("Notebook pandas:", notebook_pandas.__version__)
except ImportError as exc:
    raise RuntimeError(
        "This notebook kernel does not have pandas. From the project directory, "
        "activate .venv and run: python -m pip install -r requirements.txt"
    ) from exc


In [ ]:
import json
import urllib.request

import pandas as pd

OLLAMA = "http://localhost:11434"


def api(path, payload=None, timeout=600):
    """Call the local Ollama API (GET if no payload, POST otherwise)."""
    data = json.dumps(payload).encode() if payload is not None else None
    req = urllib.request.Request(
        OLLAMA + path, data=data, headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(req, timeout=timeout) as response:
        return json.loads(response.read())


def loaded_models():
    """Models currently in memory, with the share running on the GPU (like `ollama ps`)."""
    rows = []
    for m in api("/api/ps").get("models", []):
        size, vram = m.get("size", 0), m.get("size_vram", 0)
        rows.append({
            "model": m["name"],
            "size_GB": round(size / 1e9, 1),
            "on_GPU_%": round(100 * vram / size) if size else None,
            "context": m.get("context_length"),  # missing on older Ollama versions
        })
    return pd.DataFrame(rows)


try:
    installed = [m["name"] for m in api("/api/tags")["models"]]
    print("Ollama is running. Installed models:")
    for name in installed:
        print("  -", name)
except Exception as exc:
    print("Cannot reach Ollama at", OLLAMA, "->", exc)

## 4. Business data: calculate first, explain second

This example uses fictional sales data. Python calculates revenue, cost, gross profit, monthly totals, and product performance; the local model explains the results. **Python calculates. AI explains. You verify.** Never use the sample as real business advice.


### 4.1 Create the dedicated demo workspace and sample data

The sample data is fictional. This cell creates it in `~/local-ai-guide-sales-demo/`. The matching standalone generator is [`scripts/create_sample_sales.py`](scripts/create_sample_sales.py).

In [ ]:
from pathlib import Path

workspace = Path.home() / "local-ai-guide-sales-demo"
workspace.mkdir(parents=True, exist_ok=True)
(workspace / "Outputs").mkdir(exist_ok=True)

sample = workspace / "sample_sales.csv"
sample.write_text(
    "Month,Product,Units,UnitPrice,UnitCost\n"
    "January,Product A,100,10,6\n"
    "January,Product B,50,20,12\n"
    "February,Product A,120,10,6\n"
    "February,Product B,45,20,12\n"
    "March,Product A,150,10,6\n"
    "March,Product B,60,20,12\n",
    encoding="utf-8"
)
print("Workspace:", workspace)
print("Sample data:", sample)
print(sample.read_text(encoding="utf-8"))


### 4.2 Use the reusable analysis script

The repository includes [`analyze_sales.py`](analyze_sales.py). It validates the CSV, calculates the totals, and writes a Markdown report beside the input without modifying the CSV. Set up pandas with the instructions in [Getting started](docs/GETTING_STARTED.md).

In [ ]:
# The standalone, reusable analysis program is included as analyze_sales.py in this repository.
# It accepts any CSV path and writes its report beside the CSV.
from pathlib import Path

script = Path.cwd() / "analyze_sales.py"
if not script.is_file():
    raise FileNotFoundError("Open Jupyter with the repository directory as its working directory.")
print("Analysis script:", script)


### 4.3 Run it and verify the report

This cell runs the repository's `analyze_sales.py` with the notebook's Python interpreter. Expected totals for the sample are revenue **6,800**, cost **4,080**, and gross profit **2,720** (currency units unspecified). The complete standalone walkthrough is [Demo 03](docs/demos/03-business-sales.md).

In [ ]:
import subprocess
import sys
from pathlib import Path

workspace = Path.home() / "local-ai-guide-sales-demo"
script = Path.cwd() / "analyze_sales.py"
sample = workspace / "sample_sales.csv"
result = subprocess.run([sys.executable, str(script), str(sample)], capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("Sales analysis failed; inspect the error above.")
report = workspace / "business-analysis-report.md"
print("Report exists:", report.exists())
print(report.read_text(encoding="utf-8"))


### 4.4 Ask the local model to explain the result

Open `business-analysis-report.md` in Open WebUI or paste its content into your local chat. Ask:

> Using only this report, explain the revenue, cost, gross profit, and gross margin. Describe the monthly and product trends. Separate observations supported by the data from recommendations. Do not invent a currency, expenses, or business context that the report does not provide.

**Verify before trusting:** compare totals against Python; check for duplicate rows, totals rows, missing values, stale spreadsheet formula values, and costs not included in the file. The sample is fictional.


## 5. Level 2: VS Code + Continue

1. Install the Continue extension in VS Code while online.
2. Open the extension's configuration UI and use the current configuration format documented by Continue; configuration paths and schema can change across versions.
3. For an Ollama-backed model, select the Ollama provider and use a model tag that exactly matches `ollama list`.
4. Test a harmless task first: ask it to explain a small function before allowing edits. Review diffs and run tests.

Example configuration shape (verify against the version you install):

```yaml
name: Local Ollama
version: 1.0.0
schema: v1
models:
  - name: Llama local
    provider: ollama
    model: llama3.2:3b
    roles:
      - chat
      - edit
      - apply
```

A local model connection does not prove every extension feature is offline. Disable or avoid cloud providers and test with the network disconnected.


## 6. OpenClaw agent: file operations in an isolated demo workspace

For a complete installation and local Ollama connection, see [Install OpenClaw](docs/setup/OPENCLAW.md) and [Connect Ollama to OpenClaw](docs/connect/OLLAMA_OPENCLAW.md).

OpenClaw can use tools that read and change files. Its workspace path alone is not a security boundary. Follow [Demo 05](docs/demos/05-openclaw-files.md) to create a disposable workspace, configure and verify a Docker sandbox with restricted file tools, create and edit a sample file, and inspect each result independently. Do not give an unconfigured agent access to personal or business folders.

## 7. Advanced: performance and context length

This section is for optional discussion, not the core beginner path. Longer context windows can increase memory use, and agents may make multiple model calls per task. Actual speed depends on the model, quantization, context, GPU/CPU, runtime, and task. Treat the table below as **measurements from one machine**, not universal guarantees. Repeat the test on the target laptop before presenting the figures.

| Model | Context | Memory reported | GPU/CPU split reported |
|---|---:|---:|---|
| qwen3:8b | 4096 | 6.0 GB | 84% GPU / 16% CPU |
| qwen3:8b | 8192 | 6.6 GB | 75% GPU / 25% CPU |
| qwen3:8b | 32768 | 10 GB | 47% GPU / 53% CPU |
| llama3.2:3b | 32768 | 6.1 GB | 83% GPU / 17% CPU |

These numbers are retained from the original notebook and should be re-measured before public use. They may reflect different runtime states and are not a controlled cross-model comparison.

### Optional context sweep

Run the cell below only after confirming the model is installed and you have saved your work. It loads the model at several context sizes and may take time or consume substantial memory. Stop if the system becomes unstable.


In [ ]:
MODEL = "llama3.2:3b"
CONTEXTS = [2048, 4096, 8192, 16384, 32768]

rows = []
for ctx in CONTEXTS:
    try:
        api("/api/generate", {"model": MODEL, "keep_alive": 0})  # unload first
        api("/api/generate", {
            "model": MODEL, "prompt": "Say hi", "stream": False,
            "keep_alive": "2m", "options": {"num_ctx": ctx, "num_predict": 5},
        })
        ps = loaded_models()
        row = ps[ps["model"].str.startswith(MODEL.split(":")[0])].iloc[0]
        rows.append({"context": ctx, "size_GB": row["size_GB"], "on_GPU_%": row["on_GPU_%"]})
    except Exception as exc:
        rows.append({"context": ctx, "size_GB": None, "on_GPU_%": None, "error": str(exc)[:60]})

pd.DataFrame(rows)

### Context settings: verify per application

Different clients may pass different context sizes. Do not assume a Modelfile setting controls every client. Check the current documentation and configuration for the exact version of each client you use.

Before experimenting, back up the relevant configuration, change one setting at a time, measure the result, and keep a restore path. If a previous experiment made performance worse and you did not record the symptom, report the cause as unknown rather than guessing.

For a public demo, avoid live configuration edits. Demonstrate the prepared working configuration and keep experiments for a separate workshop.


## 8. Quick troubleshooting

| Problem | First check |
|---|---|
| Python says `ModuleNotFoundError` | Run the dependency cell with the intended notebook kernel; confirm the kernel's Python executable. |
| Ollama is unreachable | Check that Ollama is running; try `ollama list` and `curl http://127.0.0.1:11434/api/tags`. |
| Model missing | Run `ollama list`; download the model while online before the offline test. |
| Open WebUI cannot reach Ollama | Check the endpoint and whether the apps run on the host or in containers; `localhost` means different things inside a container. |
| Document answer is poor | Wait for indexing; check the selected document and retrieved passages; scanned PDFs may need OCR. |
| Offline document search fails | Verify local embeddings, parsing dependencies, and indexing are configured and available locally. |
| Model is slow | Check `ollama ps`; try a smaller model or shorter context, then re-measure. |
| Agent claims success but no output exists | Inspect the output directory and open the file; check permissions and logs. |
| Configuration edit breaks the agent | Stop the service and restore the backup; avoid changing multiple settings at once. |

## 9. Safety checklist

- Use fictional, non-sensitive data and a dedicated demo account.
- Keep passwords, tokens, personal records, and customer information out of prompts.
- Use copies of files and a dedicated agent workspace.
- Require approval for delete, overwrite, send, publish, and other consequential actions.
- Local inference does not automatically make every connected component private or offline.
- Verify outputs independently, especially calculations and agent actions.
